# Model tuning and regularization

Inspect train/validation AP, standard deviations, and fit time from the actual grid searches. Compare each model across configurations before looking at held-out accuracy.

Run the training CLI before executing this notebook.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, display

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'configs/full.json').is_file())
sys.path.insert(0, str(root))
from src.utils import read_json

pointer = root / 'models/latest.json'
if not pointer.is_file():
    raise FileNotFoundError('Run python main.py --mode train from the project root first.')
run = root / 'reports/runs' / read_json(pointer)['run_id']
metadata = read_json(run / 'metadata.json')
print(metadata['profile'], metadata['run_id'])

In [ ]:
for model in metadata['models']:
    trials = pd.read_csv(run / f'cv_{model}.csv')
    params = [column for column in trials if column.startswith('param_')]
    print(model)
    display(trials.sort_values('rank_test_score')[params + [
        'mean_train_score', 'mean_test_score', 'std_test_score', 'mean_fit_time'
    ]].head())
display(Image(filename=str(run / 'figures/mlp_loss.png')))
display(Image(filename=str(run / 'figures/learning_curves.png')))
print('For logistic regression, l1_ratio=1 means L1; l1_ratio=0 means L2.')